# 🚀 Sepsis-Hybrid-Transformer — M3 Temporal Alert Policy (M3-TAP) Advancement Notebook

**Objective:** Determine whether the high-discrimination M3 risk representation ($	ext{AUROC} = 0.9617$, $	ext{AUPRC} = 0.4231$) can be converted into **positive official PhysioNet utility** through a validation-locked temporal alert policy (M3-TAP) without modifying the held-out test predictions or leaking test information into policy selection.

### Experimental Protocol:
1. **Phase 0 Baseline Reproduction:** Verify baseline metrics on held-out test set ($N=20,000$).
2. **Phase 1 Validation Policy Sweep:** Sweep candidate temporal policies (Persistence, Hysteresis, Cooldown, SMA, EMA, Combined TAP) strictly on **Validation cohort ($N=2,034$)** to find $\pi^* = \arg\max U_{\text{val}}(\pi)$.
3. **Freeze Policy:** Lock $\pi^*$ into `m3_tap_frozen_policy.json` (zero test tuning).
4. **Phase 2 Held-Out Test Evaluation:** Evaluate frozen policy single-pass on held-out test predictions ($N=20,000$) using official PhysioNet challenge scorer.

In [ ]:
# Step 1: Clone Repository & Setup Environment
!git clone https://github.com/PravinbalajiR/Hybrid-Transform-for-Early-Sepsis-Prediction.git sepsis_app
%cd sepsis_app
!pip install numpy pandas scikit-learn matplotlib

In [ ]:
# Step 2: Phase 0 Baseline Reproduction Check
!python scripts/m3_advancement_baseline.py

In [ ]:
# Step 3: Fast Parallel Validation Temporal Policy Sweep (Phase 1)
import numpy as np
import pandas as pd
from pathlib import Path
from evaluation.utility_score import compute_utility_score
from scripts.temporal_alert_policy import (
    NaiveThresholdPolicy,
    PersistencePolicy,
    HysteresisPolicy,
    CooldownPolicy,
    MovingAveragePolicy,
    ExponentialMovingAveragePolicy,
    CombinedTAPPolicy,
)

RESULTS_DIR = Path('results')
val_data = np.load(RESULTS_DIR / 'm3_final_val_predictions.npz', allow_pickle=True)
val_y_true_flat = val_data['y_true_flat']
val_y_proba_flat = val_data['y_proba_flat']
val_lengths = val_data['patient_lengths']

val_labels, val_probs = [], []
curr = 0
for l in val_lengths:
    val_labels.append(val_y_true_flat[curr : curr + l])
    val_probs.append(val_y_proba_flat[curr : curr + l])
    curr += l

print(f'Loaded {len(val_labels):,} validation patient sequences ({len(val_y_true_flat):,} hourly records).')

In [ ]:
# Step 4: Run Validation Policy Sweep & Find Best Policy
!python scripts/run_temporal_policy_ablation.py

In [ ]:
# Step 5: Freeze Optimal Validation Policy & Evaluate Single-Pass on Test Set
import json
df_val_sweep = pd.read_csv('results/m3_validation_temporal_policy_sweep.csv')
best_row = df_val_sweep.loc[df_val_sweep['val_utility'].idxmax()]

print('=' * 80)
print('   VALIDATION UTILITY-OPTIMAL TEMPORAL POLICY SELECTED')
print('=' * 80)
print(f"Best Policy Name    : {best_row['policy_name']}")
print(f"Validation Utility  : {best_row['val_utility']:+.4f}")
print(f"Validation F1       : {best_row['val_f1']:.4f}")
print(f"Validation FPR/h    : {best_row['val_fpr_h']:.4f}")
print(f"Patient Detection   : {best_row['patient_detection_rate']*100:.1f}%")

# Save frozen policy config
frozen_config = {
    'policy_name': str(best_row['policy_name']),
    'val_utility': float(best_row['val_utility']),
    'val_f1': float(best_row['val_f1']),
    'val_fpr_h': float(best_row['val_fpr_h'])
}
with open('results/m3_tap_frozen_policy.json', 'w') as f:
    json.dump(frozen_config, f, indent=4)
print('Saved frozen policy configuration to results/m3_tap_frozen_policy.json')

In [ ]:
# Step 6: Single-Pass Evaluation of Frozen Policy on Held-Out Test Cohort (N=20,000)
test_data = np.load(RESULTS_DIR / 'm3_final_test_predictions.npz', allow_pickle=True)
test_y_true_flat = test_data['y_true_flat']
test_y_proba_flat = test_data['y_proba_flat']
test_lengths = test_data['patient_lengths']

test_labels, test_probs = [], []
curr = 0
for l in test_lengths:
    test_labels.append(test_y_true_flat[curr : curr + l])
    test_probs.append(test_y_proba_flat[curr : curr + l])
    curr += l

# Parse best policy parameters
# Evaluate on test
print('=' * 80)
print('   HELD-OUT TEST SET EVALUATION (N=20,000 PATIENTS)')
print('=' * 80)
